# 01_clean: turn raw survey codes into an analysis-ready dataset
Input: data/raw/ · Output: data/processed/clean.csv

In [9]:
import pandas as pd

# -- Load raw data --
# survey microdata: one row per respondent
df = pd.read_csv("../data/raw/polling_on_ai_usage_jul_2026_microdata.csv").copy()

# Epoch's published summary tables: used to check our cleaning
summary = pd.read_csv("../data/raw/polling_on_ai_usage_jul_2026.csv")

print("Microdata:", df.shape)
print("Summary table:", summary.shape)

Microdata: (1103, 333)
Summary table: (3008, 13)


In [10]:
# -- Split Age Groups so it matches Epoch summary table --
df["age_group"] = pd.cut(
    df["ppage"],
    bins=[17, 29, 44, 59, 120],
    labels=["18-29", "30-44", "45-59", "60+"]
)

# counts per group, in order
df["age_group"].value_counts().sort_index()

age_group
18-29    232
30-44    382
45-59    312
60+      177
Name: count, dtype: int64

In [11]:
# Check age groups against Epoch
epoch_age = summary.loc[
    (summary["demographic"] == "age_group") & (summary["question"] == "q2_services"),
    ["demographic_value", "n_base"]
].drop_duplicates() #remove repeats

epoch_age

,demographic_value,n_base
295,18–29,232
304,30–44,382
313,45–59,312
322,60+,177


✅ Age groups match Epoch's n_base exactly.

In [ ]:
# -- Education: give the codes readable levels --
edu_labels = {1: "Less than HS", 2: "High school", 3: "Some college", 4: "Bachelor+"}
df["education"] = df["ppeducat"].map(edu_labels)

# college(yes) is considered bachelors degree or higher
df["college"] = df["ppeducat"] == 4

# our counts per education label
print(df["education"].value_counts())

# Epoch count per education label
summary.loc[
    (summary["demographic"] == "education") & (summary["question"] == "q2_services"),
    ["demographic_value", "n_base"]
].drop_duplicates()

education
Bachelor+       470
High School     284
Some college    281
Less than HS     68
Name: count, dtype: int64


,demographic_value,n_base
421,Less than HS,68
430,High school,284
439,Some college,281
448,Bachelor+,470


In [14]:
# -- Income -- 
# Version 1: Epoch's 5 groups --> tables, tool, validation
income_labels = {1: "Under $25K", 2: "Under $25K", 3: "$25K–$50K", 4: "$50K–$75K",
                 5: "$75K–$100K", 6: "$100K+", 7: "$100K+"}
df["income"] = df["ppinc7"].map(income_labels)

# Version 2: same but the top group is split since it holds 57% of sample --> used for regression!
income_reg_labels = {1: "Under $25K", 2: "Under $25K", 3: "$25K–$50K", 4: "$50K–$75K",
                     5: "$75K–$100K", 6: "$100K–$150K", 7: "$150K+"}
df["income_reg"] = df["ppinc7"].map(income_reg_labels)

# -- Race/ethnicity -- 
# Epoch's 4 groups
race_labels = {1: "White, Non-Hispanic", 2: "Black, Non-Hispanic", 3: "Other/2+, Non-Hispanic",
               4: "Hispanic", 5: "Other/2+, Non-Hispanic"}
df["race"] = df["ppethm"].map(race_labels)

# -- Gender -- 
gender_labels = {1: "Male", 2: "Female"}
df["gender"] = df["ppgender"].map(gender_labels)

# Check against Epoch
for ours, theirs in [("income", "income"), ("race", "race_ethnicity"), ("gender", "gender")]:
    print("OURS:", ours)
    print(df[ours].value_counts())
    print("EPOCH:")
    print(summary.loc[
        (summary["demographic"] == theirs) & (summary["question"] == "q2_services"),
        ["demographic_value", "n_base"]
    ].drop_duplicates())
    print("-" * 40)

OURS: income
income
$100K+        634
$50K–$75K     142
$75K–$100K    139
$25K–$50K     109
Under $25K     79
Name: count, dtype: int64
EPOCH:
    demographic_value  n_base
457        Under $25K      79
466         $25K–$50K     109
475         $50K–$75K     142
484        $75K–$100K     139
493            $100K+     634
----------------------------------------
OURS: race
race
White, Non-Hispanic       666
Hispanic                  191
Black, Non-Hispanic       140
Other/2+, Non-Hispanic    106
Name: count, dtype: int64
EPOCH:
          demographic_value  n_base
349     White, Non-Hispanic     666
358     Black, Non-Hispanic     140
367                Hispanic     191
376  Other/2+, Non-Hispanic     106
----------------------------------------
OURS: gender
gender
Male      593
Female    510
Name: count, dtype: int64
EPOCH:
    demographic_value  n_base
331              Male     593
340            Female     510
----------------------------------------


In [15]:
# -- Job Groups: Q26 code --> 5 groups(overall results + regression) --
job_labels = {
    # Office/Knowledge
    1: "Office/knowledge", 30: "Office/knowledge", 31: "Office/knowledge", 3: "Office/knowledge",
    4: "Office/knowledge", 5: "Office/knowledge", 7: "Office/knowledge", 33: "Office/knowledge",
    22: "Office/knowledge",
    # Sales/service
    35: "Sales/service", 16: "Sales/service", 17: "Sales/service", 18: "Sales/service", 15: "Sales/service",
    # Manual/trades
    23: "Manual/trades", 24: "Manual/trades", 25: "Manual/trades", 26: "Manual/trades", 27: "Manual/trades",
    # Healthcare
    34: "Healthcare", 13: "Healthcare", 14: "Healthcare",
    # Education & social services
    32: "Education & social services", 6: "Education & social services",
}

# codes not in dictionary(29 other, 28 Armed forces, -1 refused) --> NaN
df["job_group"] = df["Q26"].map(job_labels)

# -- 3 Broad groups built from the 5 groups (age x job tables + tool) --
broad_labels = {
    "Office/knowledge": "Office/knowledge",
    "Healthcare": "Care & education",
    "Education & social services": "Care & education",
    "Sales/service": "Frontline",
    "Manual/trades": "Frontline",
}
df["job_broad"] = df["job_group"].map(broad_labels)

# Check
print(df["job_group"].value_counts(dropna=False))
pd.crosstab(df["job_broad"], df["age_group"])

job_group
Office/knowledge               433
Sales/service                  220
Manual/trades                  134
NaN                            120
Education & social services    104
Healthcare                      92
Name: count, dtype: int64


age_group,18-29,30-44,45-59,60+
job_broad,,,,
Care & education,43,68,57,28
Frontline,103,103,92,56
Office/knowledge,63,172,130,68
